# 03 文档嵌入模型的使用（整合优化版）

这份 Notebook 面向“可实操学习”，不仅给出 API 用法，还会解释为什么这样设计。

学习建议：
1. 先顺序跑完所有代码，确认每个阶段输入输出。
2. 再改参数做对比实验（尤其是 chunk_size、k、search_type）。
3. 最后把你自己的数据替换进来，验证迁移效果。

## 学习目标与方法

本章每个文件都采用同一套学习框架：
- 概念：这个模块在 RAG 链路里解决什么问题
- 接口：核心函数、关键参数、常见坑
- 实战：可复用代码模板 + 结果验证
- 迭代：如何优化质量、成本与稳定性

In [ ]:
from __future__ import annotations

# ========= 通用环境初始化 =========
# 说明：
# 1) 读取 .env（如果存在）
# 2) 统一工作目录与资源目录
# 3) 打印关键密钥状态，避免后面运行时报错才发现没配环境

import os
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

try:
    import dotenv
    dotenv.load_dotenv()
except Exception:
    # 没安装 python-dotenv 也不影响运行，只是不会自动加载 .env
    pass

BASE_DIR = Path.cwd()
ASSET_DIR = BASE_DIR / 'asset'
LOAD_DIR = ASSET_DIR / 'load'
assert LOAD_DIR.exists(), f'未找到数据目录: {LOAD_DIR}'

print('workdir:', BASE_DIR)
print('OPENAI_API_KEY exists:', bool(os.getenv('OPENAI_API_KEY')))
print('TAVILY_API_KEY exists:', bool(os.getenv('TAVILY_API_KEY')))

1、句子的向量化

举例：

In [1]:
from langchain_openai import OpenAIEmbeddings
import os
import dotenv

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

embedding_model = OpenAIEmbeddings(
    # model="text-embedding-ada-002"
    model = "text-embedding-3-large"
)

text = "Nice to meet you!"

embed_query = embedding_model.embed_query(text = text,)

print(len(embed_query))  # 1536 --> 3072

print(embed_query[:10])

3072
[-0.027939368039369583, 0.03950421139597893, -0.020670808851718903, -0.0008068201714195311, 0.015577414073050022, -0.00793732050806284, -0.013834580779075623, 0.01602325402200222, 0.015712516382336617, 0.06263390183448792]


2、文档的向量化

文档的向量化，接收的参数是字符串数组。

举例1：


In [4]:
from langchain_openai import OpenAIEmbeddings
import numpy as np
import pandas as pd
import os
import dotenv

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 初始化嵌入模型
embeddings_model = OpenAIEmbeddings(model="text-embedding-ada-002")

# 待嵌入的文本列表
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

# 生成嵌入向量
embeddings = embeddings_model.embed_documents(texts)


for i in range(len(texts)):
    print(f"{texts[i]}:{embeddings[i][:3]}",end="\n\n")


Hi there!:[-0.020325319841504097, -0.007096723187714815, -0.022839006036520004]

Oh, hello!:[0.004446744918823242, -0.014353534206748009, 0.0019785689655691385]

What's your name?:[-0.004887457471340895, -0.009618516080081463, 0.007236444391310215]

My friends call me World:[-0.004588752053678036, -0.014497518539428711, 0.01022490207105875]

Hello World!:[0.002393412170931697, 0.0002737858740147203, -0.0023398753255605698]



举例2：


In [5]:
from dotenv import load_dotenv
from langchain_community.document_loaders import CSVLoader
from langchain_openai import OpenAIEmbeddings


embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
)

# 情况1：
loader = CSVLoader("./asset/load/03-load.csv", encoding="utf-8")
docs = loader.load_and_split()

#print(len(docs))

# 存放的是每一个chrunk的embedding。
embeded_docs = embeddings_model.embed_documents([doc.page_content for doc in docs])
print(len(embeded_docs))
# 表示的是每一个chrunk的embedding的维度
print(len(embeded_docs[0]))
print(embeded_docs[0][:10])

4
3072
[0.0011534227523952723, 0.005336394999176264, -0.008949915878474712, 0.030998842790722847, -0.0017970810877159238, 0.011092216707766056, 0.021087471395730972, 0.048524416983127594, 0.0002677876618690789, -0.00012219828204251826]


## Optimization Add-on: cosine similarity + cache

In [ ]:
import os, math
def get_emb():
    if os.getenv('OPENAI_API_KEY'):
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model='text-embedding-3-small')
    from langchain_core.embeddings import FakeEmbeddings
    return FakeEmbeddings(size=1536)
emb=get_emb()
v1=emb.embed_query('what is ai'); v2=emb.embed_query('definition of ai'); v3=emb.embed_query('weather today')
def cos(a,b):
    s=sum(x*y for x,y in zip(a,b)); na=math.sqrt(sum(x*x for x in a)); nb=math.sqrt(sum(x*x for x in b)); return 0 if na==0 or nb==0 else s/(na*nb)
print('sim12', round(cos(v1,v2),4), 'sim13', round(cos(v1,v3),4))
class CachedEmb:
    def __init__(self, base): self.base=base; self.cache={}
    def embed_query(self, text):
        if text not in self.cache: self.cache[text]=self.base.embed_query(text)
        return self.cache[text]
cached=CachedEmb(emb); _=cached.embed_query('what is ai'); _=cached.embed_query('what is ai'); print('cache_size', len(cached.cache))

## 深度笔记：`embed_query` 与 `embed_documents` 的角色分工

- `embed_query`：用于把“用户问题”映射到向量空间
- `embed_documents`：用于把“知识片段”映射到同一向量空间

必须使用同一类 embedding 模型，才能保证距离可比较。

In [ ]:
import math

def cosine_similarity(vec_a, vec_b) -> float:
    """手写余弦相似度，帮助理解检索底层原理。"""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def explain_similarity(embeddings):
    q1 = '什么是 RAG？'
    q2 = '检索增强生成的定义是什么？'
    q3 = '上海今天会下雨吗？'
    v1 = embeddings.embed_query(q1)
    v2 = embeddings.embed_query(q2)
    v3 = embeddings.embed_query(q3)
    print('sim(q1,q2)=', round(cosine_similarity(v1, v2), 4))
    print('sim(q1,q3)=', round(cosine_similarity(v1, v3), 4))

if 'emb' in globals():
    explain_similarity(emb)
elif 'embeddings' in globals():
    explain_similarity(embeddings)
else:
    print('请先运行本文件中创建 embeddings 的代码单元。')

### 成本优化建议

1. 对重复问题做 embedding 缓存。
2. 批量调用 `embed_documents`，减少 API 往返开销。